# Plotnine — Grammar of Graphics for Python

A complete one-part tutorial for building structured, statistical, and publication-oriented figures in Python with **Plotnine**, a Grammar of Graphics implementation whose syntax is similar to **ggplot2**.

**Learning path:**

`Data → Aesthetics → Geoms/Layers → Scales → Statistics → Facets → Coordinates → Themes → Scientific Figures`

The examples use **synthetic teaching data**. They are designed to teach visualization patterns for Data Science, Bioinformatics, and Cancer AI; they are not clinical findings.

## Official references

- **Plotnine official documentation:** https://plotnine.org/
- **Plotnine guide:** https://plotnine.org/guide/
- **Plotnine geometric objects:** https://plotnine.org/guide/geometric-objects.html
- **Plotnine plot composition:** https://plotnine.org/guide/plot-composition.html
- **ggplot2 official documentation:** https://ggplot2.tidyverse.org/
- **ggplot2 Grammar of Graphics introduction:** https://ggplot2.tidyverse.org/articles/ggplot2.html
- **ggplot2 reference index:** https://ggplot2.tidyverse.org/reference/

The implementation target in this notebook is Plotnine for Python, while ggplot2 is used as the conceptual reference.

## Installation

Install Plotnine in the environment used by this notebook. For extra dependencies used by some examples, the official Plotnine documentation also provides an `extras` option.

In [49]:
# Install Plotnine from PyPI when it is not already installed.
# Run this command from a terminal or notebook cell when needed.
# !pip install plotnine

## Final function and model checklist

### Core grammar
`ggplot()`, `aes()`, `labs()`, `theme()`

### Geoms used in this notebook
`geom_point()`, `geom_line()`, `geom_path()`, `geom_bar()`, `geom_col()`, `geom_histogram()`, `geom_density()`, `geom_boxplot()`, `geom_violin()`, `geom_jitter()`, `geom_smooth()`, `geom_errorbar()`, `geom_pointrange()`, `geom_area()`, `geom_ribbon()`, `geom_text()`, `geom_label()`, `geom_hline()`, `geom_vline()`, `geom_abline()`, `geom_tile()`

### Scales and labels
`scale_x_continuous()`, `scale_y_continuous()`, `scale_x_log10()`, `scale_color_brewer()`, `scale_color_gradient()`, `scale_color_manual()`, `scale_fill_brewer()`, `scale_fill_manual()`, `labs()`

### Statistics and modeling layers
`stat_summary()`, `geom_smooth()`

### Faceting
`facet_wrap()`, `facet_grid()`

### Coordinates
`coord_cartesian()`, `coord_flip()`, `coord_fixed()`, `coord_polar()`

### Themes and figure styling
`theme_minimal()`, `theme_classic()`, `theme_bw()`, `theme_void()`, `theme()`, `element_text()`, `element_line()`, `element_rect()`

### Composition and export
Plot composition with `|` and `/`, `ggsave()`

The exact available API depends on the installed Plotnine version, so this notebook favors broadly stable public interfaces and checks version information at runtime.

## 1. Imports and teaching data

We create a synthetic clinical-style dataset with variables that resemble the structure of a research table: patient groups, age, tumor volume, gene expression, survival time, subtype, sex, and model predictions.

In [50]:
# Import the core libraries used throughout the notebook.
# Pandas stores tabular data, NumPy creates reproducible synthetic values,
# and Plotnine provides the Grammar of Graphics interface.
import numpy as np
import pandas as pd
import plotnine as p9

# Display the installed Plotnine version so the notebook can be audited later.
print("Plotnine version:", p9.__version__)

In [51]:
# Create a reproducible random number generator.
# A fixed seed makes the teaching dataset deterministic across runs.
rng = np.random.default_rng(42)

# Define sample-level categories used in the synthetic dataset.
subtypes = np.array(["Classical", "Proneural", "Mesenchymal"])
sexes = np.array(["Female", "Male"])

# Generate synthetic patient-level observations.
clinical = pd.DataFrame({
    "Patient": [f"P{i:03d}" for i in range(1, 241)],
    "Subtype": rng.choice(subtypes, size=240, p=[0.35, 0.35, 0.30]),
    "Sex": rng.choice(sexes, size=240),
    "Age": np.clip(rng.normal(58, 12, 240), 20, 90),
})

# Create synthetic biological and clinical measurements.
clinical["Tumor_Volume"] = np.clip(12 + 0.65 * clinical["Age"] + rng.normal(0, 28, 240), 5, None)
clinical["Gene_Expression"] = np.exp(rng.normal(2.1, 0.55, 240))
clinical["Survival_Days"] = np.clip(700 - 3.2 * clinical["Age"] - 0.7 * clinical["Tumor_Volume"] + rng.normal(0, 150, 240), 30, None)
clinical["Predicted_Survival"] = clinical["Survival_Days"] + rng.normal(0, 105, 240)
clinical["Risk_Score"] = np.clip((clinical["Tumor_Volume"] / 100) + rng.normal(0.45, 0.18, 240), 0, 2)

# Convert categorical columns to pandas category dtype.
# This makes the intended discrete nature of these variables explicit.
clinical["Subtype"] = clinical["Subtype"].astype("category")
clinical["Sex"] = clinical["Sex"].astype("category")

# Inspect the first rows of the teaching dataset.
clinical.head()

## 2. The core syntax: data + mapping + layer

The minimum useful mental model is:

```python
ggplot(data, aes(...)) + geom_...()
```

`ggplot()` establishes the plotting object, `aes()` maps variables to visual properties, and a `geom_` layer determines how the data is drawn.

In [52]:
# Build the simplest scatter plot from the synthetic clinical dataset.
# ggplot() initializes the plot with the data.
# aes() maps Age to the x-axis and Tumor_Volume to the y-axis.
# geom_point() draws one point for each observation.

p_basic = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Tumor_Volume"))
    + p9.geom_point()
)

p_basic

## 3. Aesthetic mappings

Aesthetics are the visual channels used to represent variables. Common mappings include `x`, `y`, `color`, `fill`, `size`, `shape`, `alpha`, `linetype`, and `group`.

A key distinction is:

- **Mapped aesthetics** go inside `aes()` because they depend on data.
- **Fixed settings** are placed outside `aes()` because they are not data mappings.

In [53]:
# Map tumor subtype to color so the biological groups are visible.
# Because Subtype is inside aes(), Plotnine creates a legend from the data.

p_color = (
    p9.ggplot(
        clinical,
        p9.aes(x="Age", y="Tumor_Volume", color="Subtype")
    )
    + p9.geom_point(size=2.6, alpha=0.75)
)

p_color

In [54]:
# Use a fixed point size and transparency instead of mapping them to variables.
# These values are styling choices, so they are intentionally outside aes().

p_fixed_style = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Tumor_Volume"))
    + p9.geom_point(size=2.8, alpha=0.65)
)

p_fixed_style

## 4. Point, line, and path geoms

These are the basic building blocks for observations and ordered trajectories.

In [55]:
# Plot individual observations with geom_point().
# This is useful when the distribution of individual samples matters.

p_points = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Gene_Expression", color="Subtype"))
    + p9.geom_point(alpha=0.65)
    + p9.labs(title="Gene expression across patient age")
)

p_points

In [56]:
# Create an ordered subset so geom_line() connects observations intentionally.
# Lines should only be used when the x-axis has a meaningful order.
trend = (
    clinical.groupby("Age", as_index=False)["Survival_Days"]
    .mean()
    .sort_values("Age")
)

# Draw the group-level trend as a line.
# group=1 treats the whole summary table as one continuous series.
p_line = (
    p9.ggplot(trend, p9.aes(x="Age", y="Survival_Days", group=1))
    + p9.geom_line(size=1.1)
    + p9.geom_point(size=2)
    + p9.labs(title="Mean survival by age")
)

p_line

In [57]:
# geom_path() follows the row order exactly rather than sorting by x.
# This is useful for trajectories, paths, or other explicitly ordered sequences.
trajectory = clinical.sort_values(["Subtype", "Age"]).iloc[:80].copy()

p_path = (
    p9.ggplot(trajectory, p9.aes(x="Age", y="Risk_Score", group="Subtype", color="Subtype"))
    + p9.geom_path(alpha=0.7)
    + p9.labs(title="Ordered risk trajectories by subtype")
)

p_path

## 5. Bars: counts versus precomputed values

`geom_bar()` is typically used when Plotnine should count observations for you. `geom_col()` is used when the heights are already present in the data.

In [58]:
# Count the number of patients in each tumor subtype automatically.
# geom_bar() performs the counting step for categorical x values.

p_bar = (
    p9.ggplot(clinical, p9.aes(x="Subtype", fill="Subtype"))
    + p9.geom_bar()
    + p9.labs(title="Patient count by tumor subtype", y="Count")
)

p_bar

In [59]:
# Precompute mean tumor volume by subtype.
# geom_col() uses the already calculated y values directly.
summary_volume = (
    clinical.groupby("Subtype", observed=True, as_index=False)["Tumor_Volume"]
    .mean()
)

# Draw the precomputed means with geom_col().
p_col = (
    p9.ggplot(summary_volume, p9.aes(x="Subtype", y="Tumor_Volume", fill="Subtype"))
    + p9.geom_col(show_legend=False)
    + p9.labs(title="Mean tumor volume by subtype", y="Mean tumor volume")
)

p_col

## 6. Distributions

Distribution plots answer different questions: histograms show binned counts, density plots show a smoothed distribution, and box/violin plots summarize distributions across groups.

In [60]:
# Show the distribution of survival time with a histogram.
# bins controls how many intervals are used to discretize the numeric variable.
# alpha improves visibility when additional layers are added later.

p_hist = (
    p9.ggplot(clinical, p9.aes(x="Survival_Days"))
    + p9.geom_histogram(bins=30, alpha=0.8)
    + p9.labs(title="Distribution of survival time", x="Survival (days)", y="Count")
)

p_hist

In [61]:
# Compare smoothed survival distributions across tumor subtypes.
# fill separates the distributions visually while alpha prevents opaque overlap.

p_density = (
    p9.ggplot(clinical, p9.aes(x="Survival_Days", fill="Subtype", color="Subtype"))
    + p9.geom_density(alpha=0.20)
    + p9.labs(title="Survival distributions by subtype")
)

p_density

In [62]:
# Use a boxplot to compare central tendency, spread, and potential outliers.
# Jittered points can be added when individual observations are important.

p_box = (
    p9.ggplot(clinical, p9.aes(x="Subtype", y="Survival_Days", fill="Subtype"))
    + p9.geom_boxplot(alpha=0.75, show_legend=False)
    + p9.labs(title="Survival distribution by subtype")
)

p_box

In [63]:
# Violin plots emphasize the shape of each group distribution.
# They are especially useful when distributions are multimodal or asymmetric.

p_violin = (
    p9.ggplot(clinical, p9.aes(x="Subtype", y="Survival_Days", fill="Subtype"))
    + p9.geom_violin(alpha=0.75, show_legend=False)
    + p9.labs(title="Violin plot of survival by subtype")
)

p_violin

In [64]:
# Jitter observations horizontally so many samples at the same category
# do not overlap at exactly the same x-position.
p_jitter = (
    p9.ggplot(clinical, p9.aes(x="Subtype", y="Survival_Days", color="Subtype"))
    + p9.geom_jitter(width=0.15, alpha=0.45, show_legend=False)
    + p9.labs(title="Individual survival observations by subtype")
)

p_jitter

In [65]:
# Combine a distribution summary with individual observations.
# This layered pattern is often clearer than using a summary plot alone.
p_box_points = (
    p9.ggplot(clinical, p9.aes(x="Subtype", y="Survival_Days", fill="Subtype"))
    + p9.geom_boxplot(alpha=0.65, show_legend=False)
    + p9.geom_jitter(width=0.13, alpha=0.35, color="black", size=1.2)
    + p9.labs(title="Boxplot with individual patient observations")
)

p_box_points

## 7. Statistical layers

Plotnine can add statistical summaries before drawing. `stat_summary()` is useful when a figure should display group summaries, while `geom_smooth()` is useful for fitted trends.

In [66]:
# Display the mean and its confidence interval for each subtype.
# stat_summary() calculates a summary statistic before drawing it.
# fun.data="mean_cl_normal" requests a mean with a normal-based confidence interval.

p_summary = (
    p9.ggplot(clinical, p9.aes(x="Subtype", y="Survival_Days"))
    + p9.stat_summary(fun_data="mean_cl_normal", geom="pointrange")
    + p9.labs(title="Mean survival with confidence intervals")
)

p_summary

## 8. Regression and trend lines

A fitted line can summarize a relationship, but it should not be mistaken for proof of causality. The goal here is visualization of association and model structure.

In [67]:
# Add a linear regression line to an age-versus-volume scatter plot.
# geom_smooth(method="lm") fits a linear model and draws the estimated trend.
# se=True displays the uncertainty band when supported by the installed version.

p_lm = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Tumor_Volume"))
    + p9.geom_point(alpha=0.55)
    + p9.geom_smooth(method="lm", se=True)
    + p9.labs(title="Linear trend: age vs tumor volume")
)

p_lm

In [68]:
# Add a nonlinear smoothing curve to visualize a potentially curved relationship.
# method="loess" requests a local smoother where supported by the environment.
# If the required statistical dependency is unavailable, use method="lm" instead.
p_loess = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Survival_Days"))
    + p9.geom_point(alpha=0.45)
    + p9.geom_smooth(method="loess", se=False)
    + p9.labs(title="Nonlinear trend: age vs survival")
)

p_loess

## 9. Error bars and point ranges

Error bars are useful when the uncertainty interval itself is the focus. They should always be paired with a clear definition of what the interval represents.

In [ ]:
# Calculate the group mean and a simple normal-approximation confidence interval.
# This is only a teaching summary; it is not a substitute for a study-specific analysis plan.
group_summary = (
    clinical.groupby("Subtype", observed=True)["Tumor_Volume"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
group_summary["se"] = group_summary["std"] / np.sqrt(group_summary["count"])
group_summary["lower"] = group_summary["mean"] - 1.96 * group_summary["se"]
group_summary["upper"] = group_summary["mean"] + 1.96 * group_summary["se"]

# Draw group means as points and confidence intervals as vertical error bars.
p_errorbar = (
    p9.ggplot(group_summary, p9.aes(x="Subtype", y="mean"))
    + p9.geom_point(size=3)
    + p9.geom_errorbar(p9.aes(ymin="lower", ymax="upper"), width=0.18)
    + p9.labs(title="Mean tumor volume with 95% confidence intervals", y="Mean tumor volume")
)

p_errorbar

## 10. Scales

Scales translate data values into visual properties. This includes axes, colors, fills, and transformations. In Grammar of Graphics terms, scales are not decoration only; they control the data-to-visual mapping.

In [ ]:
# Change the x-axis labels and visible limits with a continuous scale.
# breaks controls the tick positions, and limits focuses the displayed range.
p_scale_axis = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Tumor_Volume"))
    + p9.geom_point(alpha=0.55)
    + p9.scale_x_continuous(breaks=[20, 40, 60, 80, 90])
    + p9.labs(title="Custom continuous x-axis scale", x="Age (years)", y="Tumor volume")
)

p_scale_axis

In [69]:
# Use a logarithmic x-axis for a strongly skewed gene-expression variable.
# scale_x_log10() is useful when multiplicative differences are more meaningful than additive ones.
p_scale_log = (
    p9.ggplot(clinical, p9.aes(x="Gene_Expression", y="Survival_Days", color="Subtype"))
    + p9.geom_point(alpha=0.65)
    + p9.scale_x_log10()
    + p9.labs(title="Log-scaled gene expression vs survival")
)

p_scale_log

In [70]:
# Apply a qualitative Brewer palette to a categorical color mapping.
# This is appropriate when categories do not have an inherent numeric order.
p_brewer = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Tumor_Volume", color="Subtype"))
    + p9.geom_point(size=2.5, alpha=0.7)
    + p9.scale_color_brewer(type="qual", palette="Set2")
    + p9.labs(title="Categorical color scale")
)

p_brewer

In [71]:
# Define an explicit categorical palette so the same group colors can be reused.
# Manual scales are valuable when figure consistency matters across multiple panels.
subtype_colors = {
    "Classical": "#4C78A8",
    "Proneural": "#F58518",
    "Mesenchymal": "#54A24B",
}

p_manual = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Tumor_Volume", color="Subtype"))
    + p9.geom_point(size=2.5, alpha=0.7)
    + p9.scale_color_manual(values=subtype_colors)
    + p9.labs(title="Reusable manual group colors")
)

p_manual

## 11. Labels and annotations

Research figures need precise titles, axis labels, captions, and sometimes explicit threshold annotations.

In [72]:
# Add a title, subtitle, caption, and clearer axis labels.
# labs() centralizes the descriptive metadata of a figure.
p_labels = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Tumor_Volume", color="Subtype"))
    + p9.geom_point(alpha=0.65)
    + p9.labs(
        title="Tumor volume across patient age",
        subtitle="Synthetic teaching data grouped by molecular subtype",
        x="Age (years)",
        y="Tumor volume (arbitrary units)",
        color="Molecular subtype",
        caption="Educational example — not clinical data"
    )
)

p_labels

In [73]:
# Add a horizontal reference line to indicate a teaching threshold.
# geom_hline() adds a fixed y-axis reference without changing the data mapping.
p_hline = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Risk_Score"))
    + p9.geom_point(alpha=0.55)
    + p9.geom_hline(yintercept=1.0, linetype="dashed")
    + p9.labs(title="Risk score with a reference threshold")
)

p_hline

In [74]:
# Add a vertical threshold and a diagonal identity line to a prediction plot.
# geom_vline() marks an x-axis threshold, while geom_abline() adds y = x when both intercept and slope are 0 and 1.
p_prediction_reference = (
    p9.ggplot(clinical, p9.aes(x="Survival_Days", y="Predicted_Survival"))
    + p9.geom_point(alpha=0.6)
    + p9.geom_abline(intercept=0, slope=1, linetype="dashed")
    + p9.labs(title="Predicted vs actual survival", x="Actual survival (days)", y="Predicted survival (days)")
)

p_prediction_reference

## 12. Faceting

Faceting creates small multiples. It is often better than forcing too many categories into one crowded panel.

In [75]:
# Create one panel per tumor subtype.
# facet_wrap() is useful when a single categorical variable defines the panels.
p_facet_wrap = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Survival_Days"))
    + p9.geom_point(alpha=0.55)
    + p9.facet_wrap("~Subtype")
    + p9.labs(title="Survival-age relationship by subtype")
)

p_facet_wrap

In [76]:
# Create a two-dimensional grid using subtype and sex.
# facet_grid() is useful when two categorical dimensions define the experimental layout.
p_facet_grid = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Tumor_Volume"))
    + p9.geom_point(alpha=0.55)
    + p9.facet_grid("Sex ~ Subtype")
    + p9.labs(title="Tumor volume by sex and molecular subtype")
)

p_facet_grid

## 13. Coordinates

Coordinate systems control how x and y positions are interpreted and displayed. They are especially useful for horizontal categorical figures and fixed aspect ratios.

In [77]:
# Flip the Cartesian coordinates to produce a horizontal categorical chart.
# Horizontal layouts can improve readability when category labels are long.
p_flip = (
    p9.ggplot(summary_volume, p9.aes(x="Subtype", y="Tumor_Volume", fill="Subtype"))
    + p9.geom_col(show_legend=False)
    + p9.coord_flip()
    + p9.labs(title="Horizontal comparison of mean tumor volume")
)

p_flip

In [78]:
# Use coord_cartesian() to zoom the visual window without dropping data from the statistical computation.
p_zoom = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Tumor_Volume"))
    + p9.geom_point(alpha=0.55)
    + p9.coord_cartesian(ylim=(0, 180))
    + p9.labs(title="Zoomed visual window")
)

p_zoom

## 14. Themes and publication-oriented styling

Themes control non-data elements such as grids, text, axes, legends, and figure background. A publication-style figure usually benefits from restrained non-data ink, consistent typography, and enough white space.

In [79]:
# Start with a clean theme designed for a simple scientific figure.
# theme_classic() removes most background grid lines and leaves the axes prominent.
p_theme_classic = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Tumor_Volume", color="Subtype"))
    + p9.geom_point(size=2.5, alpha=0.7)
    + p9.theme_classic()
    + p9.labs(title="Clean scientific scatter plot")
)

p_theme_classic

In [80]:
# Customize typography and spacing with theme().
# element_text() controls text appearance, while plot margins create breathing room around the figure.
p_pub = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Tumor_Volume", color="Subtype"))
    + p9.geom_point(size=2.4, alpha=0.7)
    + p9.theme_classic()
    + p9.theme(
        figure_size=(6.5, 4.5),
        plot_title=p9.element_text(size=14, weight="bold"),
        axis_title=p9.element_text(size=11),
        axis_text=p9.element_text(size=9),
        legend_title=p9.element_text(size=10),
        legend_text=p9.element_text(size=9),
        plot_margin=0.12,
    )
    + p9.labs(title="Publication-oriented figure template")
)

p_pub

## 15. Bioinformatics: gene expression visualization

A common scientific pattern is to compare a continuous expression measurement across biological groups. A boxplot plus jittered observations gives both a summary and a view of sample-level variability.

In [81]:
# Compare synthetic gene expression across molecular subtypes.
# The log transformation on the y-axis makes a right-skewed expression variable easier to inspect.
p_gene_box = (
    p9.ggplot(clinical, p9.aes(x="Subtype", y="Gene_Expression", fill="Subtype"))
    + p9.geom_boxplot(alpha=0.65, show_legend=False)
    + p9.geom_jitter(width=0.12, alpha=0.35, size=1.2, color="black")
    + p9.scale_y_log10()
    + p9.theme_classic()
    + p9.labs(
        title="Gene expression by molecular subtype",
        x="Molecular subtype",
        y="Gene expression (log10 scale)"
    )
)

p_gene_box

## 16. Bioinformatics: expression-style heatmap

Heatmaps are useful for compactly displaying many measurements across samples or genes. Here we create synthetic gene-level measurements and reshape them into a tidy table.

In [82]:
# Create a synthetic gene-expression matrix for a small teaching example.
genes = [f"Gene_{i:02d}" for i in range(1, 16)]
samples = [f"S{i:02d}" for i in range(1, 13)]

expr_matrix = pd.DataFrame(
    rng.normal(0, 1, size=(len(genes), len(samples))),
    index=genes,
    columns=samples,
)

# Convert the matrix to long form because Plotnine works naturally with tidy data.
expr_long = (
    expr_matrix
    .reset_index(names="Gene")
    .melt(id_vars="Gene", var_name="Sample", value_name="Expression_Z")
)

expr_long.head()

In [83]:
# Map gene, sample, and expression value into a tile-based heatmap.
# geom_tile() draws one rectangle per row of the tidy expression table.
# A diverging fill scale is appropriate for values centered around zero.
p_heatmap = (
    p9.ggplot(expr_long, p9.aes(x="Sample", y="Gene", fill="Expression_Z"))
    + p9.geom_tile()
    + p9.scale_fill_gradient2(low="#3B4CC0", mid="white", high="#B40426", midpoint=0)
    + p9.theme_minimal()
    + p9.theme(axis_text_x=p9.element_text(angle=45, ha="right"))
    + p9.labs(title="Synthetic gene-expression heatmap", fill="Expression Z-score")
)

p_heatmap

## 17. Cancer AI: predicted vs actual

Prediction-vs-actual plots are useful for regression models. The identity line represents perfect agreement; the scatter shows how far individual predictions deviate from it.

In [84]:
# Calculate prediction errors for the synthetic survival model.
# Residuals are defined here as predicted minus actual survival.
clinical["Residual"] = clinical["Predicted_Survival"] - clinical["Survival_Days"]

# Draw predicted values against actual values with an identity reference line.
p_ai_prediction = (
    p9.ggplot(
        clinical,
        p9.aes(x="Survival_Days", y="Predicted_Survival", color="Subtype")
    )
    + p9.geom_point(alpha=0.65)
    + p9.geom_abline(intercept=0, slope=1, linetype="dashed")
    + p9.theme_classic()
    + p9.labs(
        title="Cancer AI regression: predicted vs actual survival",
        x="Actual survival (days)",
        y="Predicted survival (days)",
        color="Subtype",
    )
)

p_ai_prediction

In [85]:
# Visualize the residual distribution by tumor subtype.
# A centered residual distribution with small spread would indicate closer agreement.
p_ai_residual = (
    p9.ggplot(clinical, p9.aes(x="Subtype", y="Residual", fill="Subtype"))
    + p9.geom_boxplot(alpha=0.7, show_legend=False)
    + p9.geom_hline(yintercept=0, linetype="dashed")
    + p9.theme_classic()
    + p9.labs(
        title="Prediction residuals by subtype",
        x="Tumor subtype",
        y="Residual (predicted − actual days)",
    )
)

p_ai_residual

## 18. Bioinformatics: volcano-style differential-expression figure

A volcano plot commonly places effect size on the x-axis and a negative log10 significance measure on the y-axis. This example is only a visualization template; significance thresholds and multiple-testing procedures must be defined by the actual study.

In [86]:
# Create synthetic differential-expression-style statistics.
# log2_fc represents effect size and p_value represents a teaching significance measure.
volcano = pd.DataFrame({
    "Gene": [f"Gene_{i:03d}" for i in range(1, 501)],
    "log2_fc": rng.normal(0, 1.1, 500),
    "p_value": 10 ** (-rng.uniform(0.2, 8, 500)),
})

# Transform p-values so smaller values appear higher on the plot.
volcano["minus_log10_p"] = -np.log10(volcano["p_value"])

# Define simple teaching thresholds for highlighting points.
volcano["Status"] = "Not significant"
volcano.loc[(volcano["log2_fc"] >= 1) & (volcano["p_value"] < 0.05), "Status"] = "Up"
volcano.loc[(volcano["log2_fc"] <= -1) & (volcano["p_value"] < 0.05), "Status"] = "Down"

# Draw a volcano-style plot with fold-change and significance thresholds.
p_volcano = (
    p9.ggplot(volcano, p9.aes(x="log2_fc", y="minus_log10_p", color="Status"))
    + p9.geom_point(alpha=0.7, size=1.8)
    + p9.geom_vline(xintercept=[-1, 1], linetype="dashed")
    + p9.geom_hline(yintercept=-np.log10(0.05), linetype="dashed")
    + p9.theme_classic()
    + p9.labs(
        title="Volcano-style differential-expression figure",
        x="log2 fold change",
        y="−log10(p-value)",
        color="Classification",
    )
)

p_volcano

## 19. Survival-oriented visualization

This section is intentionally limited to exploratory visualization. A true Kaplan–Meier analysis requires survival-analysis methodology and appropriate censoring information; Plotnine is the visualization layer, not the survival-analysis engine.

In [87]:
# Bin age into clinically interpretable teaching groups.
# The bins are arbitrary for demonstration and should not be treated as clinical cutoffs.
clinical["Age_Group"] = pd.cut(
    clinical["Age"],
    bins=[0, 45, 60, 75, 120],
    labels=["<45", "45–60", "60–75", "75+"],
    right=False,
)

# Compare survival distributions across age groups.
p_survival_age = (
    p9.ggplot(clinical, p9.aes(x="Age_Group", y="Survival_Days", fill="Age_Group"))
    + p9.geom_violin(alpha=0.65, show_legend=False)
    + p9.geom_boxplot(width=0.15, alpha=0.9, show_legend=False)
    + p9.theme_classic()
    + p9.labs(title="Exploratory survival distribution by age group", x="Age group", y="Survival (days)")
)

p_survival_age

## 20. Missing-data visualization

Missingness can be treated as a data-quality variable. A simple tile map makes patterns across variables and samples visible.

In [88]:
# Create a small synthetic clinical table with intentional missing values.
missing_demo = clinical[["Age", "Tumor_Volume", "Gene_Expression", "Survival_Days", "Predicted_Survival"]].copy()
missing_demo.loc[rng.choice(missing_demo.index, 25, replace=False), "Gene_Expression"] = np.nan
missing_demo.loc[rng.choice(missing_demo.index, 18, replace=False), "Tumor_Volume"] = np.nan

# Convert the missingness matrix to tidy form.
missing_long = (
    missing_demo.isna()
    .head(80)
    .reset_index(names="Row")
    .melt(id_vars="Row", var_name="Variable", value_name="Missing")
)

# Map missing and observed cells as two discrete colors.
p_missing = (
    p9.ggplot(missing_long, p9.aes(x="Variable", y="Row", fill="Missing"))
    + p9.geom_tile()
    + p9.scale_fill_manual(values={False: "white", True: "#333333"})
    + p9.theme_minimal()
    + p9.theme(axis_text_x=p9.element_text(angle=45, ha="right"), axis_text_y=p9.element_blank())
    + p9.labs(title="Synthetic missing-data pattern", x="Variable", y="Sample row", fill="Missing")
)

p_missing

## 21. Faceted research figure

Faceting becomes powerful when the same relationship must be inspected across multiple biological or clinical groups.

In [89]:
# Compare age and tumor volume across sex and subtype using small multiples.
# The same coordinate system is reused across all panels to make visual comparisons easier.
p_research_facet = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Tumor_Volume", color="Sex"))
    + p9.geom_point(alpha=0.55)
    + p9.geom_smooth(method="lm", se=False)
    + p9.facet_wrap("~Subtype")
    + p9.theme_classic()
    + p9.labs(
        title="Tumor volume across age by molecular subtype",
        subtitle="Sex is represented by point color; lines show linear trends",
    )
)

p_research_facet

## 22. Area and ribbon layers

`geom_area()` is useful for cumulative or stacked quantities. `geom_ribbon()` is especially useful for uncertainty bands around a line.

In [90]:
# Build a smooth teaching curve and an uncertainty interval.
curve = pd.DataFrame({
    "Age": np.linspace(20, 90, 80),
})
curve["Mean_Survival"] = 760 - 4.0 * curve["Age"] + 0.02 * (curve["Age"] - 55) ** 2
curve["Lower"] = curve["Mean_Survival"] - 70
curve["Upper"] = curve["Mean_Survival"] + 70

# Use geom_ribbon() for the uncertainty region and geom_line() for the center line.
p_ribbon = (
    p9.ggplot(curve, p9.aes(x="Age", y="Mean_Survival"))
    + p9.geom_ribbon(p9.aes(ymin="Lower", ymax="Upper"), alpha=0.25)
    + p9.geom_line(size=1.1)
    + p9.theme_classic()
    + p9.labs(title="Mean trend with an uncertainty band", x="Age (years)", y="Estimated survival (days)")
)

p_ribbon

## 23. Publication-style annotations and panel tags

Panel tags such as `A`, `B`, and `C` are common in manuscripts. Plotnine supports labels and tags through the plot labeling system and theme controls.

In [91]:
# Add a publication-style panel tag with labs(tag=...).
# The tag can be positioned with theme(plot_tag_position=...) in current Plotnine versions.
p_tag = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Tumor_Volume"))
    + p9.geom_point(alpha=0.55)
    + p9.theme_classic()
    + p9.theme(plot_tag_position="top-left")
    + p9.labs(tag="A", title="Panel A: age vs tumor volume")
)

p_tag

## 24. Plot composition

One of Plotnine's useful features is composition of multiple plots. Current Plotnine documentation supports operators such as `|` and `/` for arranging plots side-by-side or vertically.

In [92]:
# Build three small figures that share the same teaching dataset.
p1 = (
    p9.ggplot(clinical, p9.aes(x="Age", y="Tumor_Volume"))
    + p9.geom_point(alpha=0.55)
    + p9.theme_classic()
    + p9.labs(tag="A", title="Age vs volume")
)

p2 = (
    p9.ggplot(clinical, p9.aes(x="Subtype", y="Survival_Days", fill="Subtype"))
    + p9.geom_boxplot(show_legend=False)
    + p9.theme_classic()
    + p9.labs(tag="B", title="Survival by subtype")
)

p3 = (
    p9.ggplot(clinical, p9.aes(x="Survival_Days", y="Predicted_Survival"))
    + p9.geom_point(alpha=0.55)
    + p9.geom_abline(intercept=0, slope=1, linetype="dashed")
    + p9.theme_classic()
    + p9.labs(tag="C", title="Prediction agreement")
)

# Place the first two plots side by side and stack the third below them.
# This follows Plotnine's composition syntax documented in the official guide.
composed_figure = (p1 | p2) / p3

composed_figure

## 25. Exporting a figure

For a manuscript, save figures deliberately with explicit dimensions and resolution. Plotnine provides `ggsave()` for ggplot-style objects.

In [93]:
# Create a publication-ready output directory.
# The directory is kept local to the notebook project.
from pathlib import Path
output_dir = Path("figures")
output_dir.mkdir(exist_ok=True)

# Save the publication-oriented figure as a high-resolution PNG.
# width and height are measured in inches in the Plotnine/Matplotlib workflow.
# dpi controls raster output resolution.
# Uncomment the next line when you want to export the figure.
# p_pub.save(output_dir / "tumor_volume_publication.png", width=6.5, height=4.5, dpi=300)

# A vector PDF is often preferable for line art and text when the journal accepts it.
# Uncomment the next line when a PDF export is required.
# p_pub.save(output_dir / "tumor_volume_publication.pdf", width=6.5, height=4.5)

## 26. Mini project — publication figure workflow

The following example combines data, mapping, geometry, statistical smoothing, faceting, labels, a manual palette, and a restrained theme. This is the pattern to imitate when building a manuscript figure rather than writing a one-off chart.

In [94]:
# Build a complete research-style figure from several Grammar of Graphics components.
# Start with tidy data and map the variables that matter scientifically.
final_publication = (
    p9.ggplot(
        clinical,
        p9.aes(x="Age", y="Tumor_Volume", color="Subtype")
    )
    + p9.geom_point(size=2.2, alpha=0.68)
    + p9.geom_smooth(method="lm", se=False, linetype="solid")
    + p9.facet_wrap("~Sex")
    + p9.scale_color_manual(values=subtype_colors)
    + p9.theme_classic()
    + p9.theme(
        figure_size=(7.2, 4.7),
        plot_title=p9.element_text(size=14, weight="bold"),
        axis_title=p9.element_text(size=10),
        axis_text=p9.element_text(size=9),
        strip_text=p9.element_text(size=10, weight="bold"),
        legend_title=p9.element_text(size=10),
        legend_text=p9.element_text(size=9),
        plot_margin=0.12,
    )
    + p9.labs(
        title="Tumor volume and age across patient groups",
        subtitle="Synthetic example illustrating a reproducible research-figure workflow",
        x="Age (years)",
        y="Tumor volume (arbitrary units)",
        color="Molecular subtype",
        caption="Educational visualization only — synthetic data",
    )
)

final_publication

## 27. Practical decision guide

### Use `geom_point()`
When individual observations and relationships are the main question.

### Use `geom_line()`
When observations have a meaningful order such as time or a sequence.

### Use `geom_bar()`
When Plotnine should count categorical observations.

### Use `geom_col()`
When you already calculated the heights.

### Use `geom_histogram()`
When you want a binned distribution.

### Use `geom_density()`
When you want a smoothed distribution.

### Use `geom_boxplot()`
When comparing medians, spread, and outliers across groups.

### Use `geom_violin()`
When the distribution shape itself is informative.

### Use `geom_jitter()`
When categorical observations overlap and sample-level visibility matters.

### Use `geom_smooth()`
When a fitted or smoothed trend is useful for exploratory interpretation.

### Use `facet_wrap()` / `facet_grid()`
When multiple groups need comparable small-multiple panels.

### Use `geom_tile()`
When a rectangular matrix or feature-by-sample structure should be visualized.

### Use `coord_flip()`
When long categorical labels are easier to read horizontally.

### Use a composed figure
When a manuscript figure needs several coordinated panels rather than one overloaded plot.

## 28. Golden rules for scientific visualization

1. **Start from the research question, not the chart type.**
2. **Keep the data tidy** so mappings remain explicit and reproducible.
3. **Put variables inside `aes()` when the visual property is data-driven.**
4. **Keep fixed styling outside `aes()`.**
5. **Use facets when comparisons are easier as small multiples.**
6. **Do not hide the raw observations when the sample size is small.**
7. **Define uncertainty explicitly** — confidence interval, standard error, prediction interval, or another quantity.
8. **Use consistent scales and group colors across related panels.**
9. **Prefer vector output or high-resolution raster output for manuscripts.**
10. **A publication-looking figure is not automatically a scientifically valid figure.** The statistical method, study design, sample definition, and reporting standard still determine whether a figure is appropriate.

## References

**Plotnine**
https://plotnine.org/
https://plotnine.org/guide/
https://plotnine.org/guide/geometric-objects.html
https://plotnine.org/guide/plot-composition.html

**ggplot2**
https://ggplot2.tidyverse.org/
https://ggplot2.tidyverse.org/articles/ggplot2.html
https://ggplot2.tidyverse.org/reference/

These official references should be treated as the source of truth for version-specific behavior and the complete API.